# Model Evaluation Notebook

This notebook provides tools for evaluating the NLP models and retrieval quality.

In [ ]:
# Import required libraries
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import precision_recall_curve, average_precision_score

plt.style.use('seaborn-v0_8-whitegrid')

print('Libraries loaded successfully!')

## 1. Entity Recognition Evaluation

In [ ]:
from backend.core.nlp import EntityExtractor

# Initialize entity extractor
entity_extractor = EntityExtractor(use_scispacy=False)

# Sample test data with ground truth
test_data = [
    {
        'text': 'Geoffrey Hinton developed deep learning at the University of Toronto.',
        'ground_truth': [
            {'text': 'Geoffrey Hinton', 'type': 'PERSON'},
            {'text': 'University of Toronto', 'type': 'ORG'},
        ]
    },
    {
        'text': 'OpenAI released GPT-4 in March 2023.',
        'ground_truth': [
            {'text': 'OpenAI', 'type': 'ORG'},
            {'text': 'GPT-4', 'type': 'PRODUCT'},
            {'text': 'March 2023', 'type': 'DATE'},
        ]
    },
]

print(f'Test dataset: {len(test_data)} samples')

In [ ]:
def evaluate_ner(extractor, test_data):
    """Evaluate named entity recognition."""
    results = []
    
    for sample in test_data:
        # Extract entities
        predicted = extractor.extract(sample['text'], 'test')
        pred_texts = {e.text for e in predicted}
        true_texts = {e['text'] for e in sample['ground_truth']}
        
        # Calculate metrics
        tp = len(pred_texts & true_texts)
        fp = len(pred_texts - true_texts)
        fn = len(true_texts - pred_texts)
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        results.append({
            'text': sample['text'][:50] + '...',
            'true_entities': len(true_texts),
            'pred_entities': len(pred_texts),
            'precision': precision,
            'recall': recall,
            'f1': f1,
        })
    
    return pd.DataFrame(results)

ner_results = evaluate_ner(entity_extractor, test_data)
ner_results

## 2. Retrieval Quality Evaluation

In [ ]:
def evaluate_retrieval(queries, relevant_docs, retrieved_docs, k=10):
    """Evaluate retrieval quality using standard IR metrics."""
    metrics = {
        'precision_at_k': [],
        'recall_at_k': [],
        'ndcg_at_k': [],
        'mrr': [],
    }
    
    for query, relevant, retrieved in zip(queries, relevant_docs, retrieved_docs):
        retrieved_k = retrieved[:k]
        
        # Precision@K
        relevant_retrieved = len(set(retrieved_k) & set(relevant))
        precision = relevant_retrieved / k
        metrics['precision_at_k'].append(precision)
        
        # Recall@K
        recall = relevant_retrieved / len(relevant) if relevant else 0
        metrics['recall_at_k'].append(recall)
        
        # MRR
        mrr = 0
        for i, doc in enumerate(retrieved_k):
            if doc in relevant:
                mrr = 1 / (i + 1)
                break
        metrics['mrr'].append(mrr)
        
        # NDCG@K
        dcg = sum((1 if doc in relevant else 0) / np.log2(i + 2) for i, doc in enumerate(retrieved_k))
        ideal_retrieved = sorted(retrieved_k, key=lambda x: x in relevant, reverse=True)
        idcg = sum((1 if doc in relevant else 0) / np.log2(i + 2) for i, doc in enumerate(ideal_retrieved))
        ndcg = dcg / idcg if idcg > 0 else 0
        metrics['ndcg_at_k'].append(ndcg)
    
    return {k: np.mean(v) for k, v in metrics.items()}

# Example evaluation
sample_queries = ['machine learning', 'deep learning']
sample_relevant = [['doc1', 'doc2'], ['doc3', 'doc4']]
sample_retrieved = [['doc1', 'doc5', 'doc2', 'doc6'], ['doc3', 'doc7', 'doc8', 'doc4']]

retrieval_metrics = evaluate_retrieval(sample_queries, sample_relevant, sample_retrieved, k=4)
print('Retrieval Metrics:')
for metric, value in retrieval_metrics.items():
    print(f'  {metric}: {value:.4f}')

## 3. Embedding Quality Evaluation

In [ ]:
from backend.core.nlp import EmbeddingsManager
from sklearn.metrics.pairwise import cosine_similarity

# Initialize embeddings manager
embeddings_manager = EmbeddingsManager()

# Test semantic similarity
test_pairs = [
    ('Machine learning is a subset of AI.', 'AI includes machine learning.', True),
    ('Deep learning uses neural networks.', 'Neural networks power deep learning.', True),
    ('Natural language processing analyzes text.', 'The weather is nice today.', False),
]

print('Semantic Similarity Tests:')
for text1, text2, should_be_similar in test_pairs:
    emb1 = embeddings_manager.embed_query(text1)
    emb2 = embeddings_manager.embed_query(text2)
    
    similarity = cosine_similarity([emb1], [emb2])[0][0]
    
    status = '✓' if (similarity > 0.5) == should_be_similar else '✗'
    print(f'{status} Similarity: {similarity:.4f} (expected: {"high" if should_be_similar else "low"})')
    print(f'   "{text1[:40]}..." vs "{text2[:40]}..."')

## 4. Model Performance Visualization

In [ ]:
def plot_metrics_comparison(metrics_dict, title='Model Performance Comparison'):
    """Plot comparison of multiple metrics."""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    models = list(metrics_dict.keys())
    metrics = list(metrics_dict[models[0]].keys())
    
    x = np.arange(len(metrics))
    width = 0.8 / len(models)
    
    for i, model in enumerate(models):
        values = [metrics_dict[model][m] for m in metrics]
        ax.bar(x + i * width, values, width, label=model)
    
    ax.set_xlabel('Metrics')
    ax.set_ylabel('Score')
    ax.set_title(title)
    ax.set_xticks(x + width * (len(models) - 1) / 2)
    ax.set_xticklabels(metrics, rotation=45, ha='right')
    ax.legend()
    ax.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.show()

# Sample comparison data
sample_metrics = {
    'Base Model': {'precision': 0.75, 'recall': 0.70, 'f1': 0.72},
    'Fine-tuned': {'precision': 0.85, 'recall': 0.82, 'f1': 0.83},
    'Ensemble': {'precision': 0.88, 'recall': 0.85, 'f1': 0.86},
}

plot_metrics_comparison(sample_metrics, 'NER Model Comparison')

## 5. Confusion Matrix Analysis

In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels, title='Confusion Matrix'):
    """Plot confusion matrix heatmap."""
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Sample entity type classification
y_true = ['PERSON', 'ORG', 'PERSON', 'CONCEPT', 'ORG', 'CONCEPT', 'PERSON', 'ORG']
y_pred = ['PERSON', 'ORG', 'ORG', 'CONCEPT', 'ORG', 'PERSON', 'PERSON', 'ORG']
labels = ['PERSON', 'ORG', 'CONCEPT']

plot_confusion_matrix(y_true, y_pred, labels, 'Entity Type Classification')

## 6. Export Evaluation Report

In [ ]:
import json
from datetime import datetime

def generate_evaluation_report(ner_results, retrieval_metrics):
    """Generate comprehensive evaluation report."""
    report = {
        'timestamp': datetime.now().isoformat(),
        'ner_evaluation': {
            'avg_precision': ner_results['precision'].mean(),
            'avg_recall': ner_results['recall'].mean(),
            'avg_f1': ner_results['f1'].mean(),
            'samples_evaluated': len(ner_results),
        },
        'retrieval_evaluation': retrieval_metrics,
        'summary': 'Evaluation completed successfully.',
    }
    
    return report

report = generate_evaluation_report(ner_results, retrieval_metrics)
print(json.dumps(report, indent=2, default=str))

# Save report
# with open('../data/exports/evaluation_report.json', 'w') as f:
#     json.dump(report, f, indent=2, default=str)